# Практическая работа 3
## Работа с реальными данными
**Цель:** Научиться считывать и объединять CSV и Excel файлы, управлять типами данных, выполнять агрегацию (выручка по категориям) и сохранять результат в формате `parquet`.

In [1]:
%pip install openpyxl
%pip install pyarrow

### Задание 1. Считать CSV-файлы заказов и Excel-файл с товарами.

In [2]:
import pandas as pd
import glob
import os

data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)

# 1.1 Считываем все CSV файлы с заказами
csv_files = glob.glob(f'{data_dir}/orders_2024_*.csv')

if csv_files:
    # Читаем каждый файл и объединяем их в один DataFrame
    df_orders = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)
    print(f"Заказы загружены. Всего строк: {len(df_orders)}")
else:
    print("Файлы заказов не найдены. Убедитесь, что они лежат в папке data/")
    # Создаем пустой DataFrame с нужными колонками для избежания ошибок ниже
    df_orders = pd.DataFrame(columns=['order_id', 'user_id', 'product_id', 'price', 'quantity'])

# 1.2 Считываем справочник товаров
try:
    df_products = pd.read_excel(f'{data_dir}/products.xlsx')
    print(f"Товары загружены. Всего строк: {len(df_products)}")
except FileNotFoundError:
    print("Файл products.xlsx не найден в папке data/")
    df_products = pd.DataFrame(columns=['product_id', 'category_id', 'product_name'])

display(df_orders.head(3))
display(df_products.head(3))

Заказы загружены. Всего строк: 15000
Товары загружены. Всего строк: 30


,order_id,user_id,order_datetime,product_id,price,quantity,status
0,1,1044,2024-01-03 18:39:26,26,14038.15,1,refunded
1,2,1079,2024-01-03 17:45:43,16,9172.60,1,paid
2,3,1041,2024-01-06 22:46:38,25,9035.26,2,paid


,product_id,product_name,category_id,category_name,weight_kg
0,1,Product 1,1,Electronics,11.55
1,2,Product 2,3,Sport,2.22
2,3,Product 3,5,Toys,11.43


### Задание 2. Объединить заказы и товары, привести цены к типу float32, а идентификаторы к int32.

In [3]:
# Объединяем таблицы по ключу product_id (Left Join)
df_merged = df_orders.merge(df_products, on='product_id', how='left')

# Приводим столбец с ценами к float32
if 'price' in df_merged.columns:
    df_merged['price'] = df_merged['price'].astype('float32')

# Приводим все идентификаторы к int32
# (Для конвертации в стандартный int32 пропусков быть не должно, поэтому заполняем их 0)
id_cols = ['order_id', 'user_id', 'product_id', 'category_id']
for col in id_cols:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].fillna(0).astype('int32')

print("Типы данных после преобразования:")
print(df_merged.dtypes)
print()
display(df_merged.head(3))

Типы данных после преобразования:
order_id            int32
user_id             int32
order_datetime        str
product_id          int32
price             float32
quantity            int64
status                str
product_name          str
category_id         int32
category_name         str
weight_kg         float64
dtype: object



,order_id,user_id,order_datetime,product_id,price,quantity,status,product_name,category_id,category_name,weight_kg
0,1,1044,2024-01-03 18:39:26,26,14038.150391,1,refunded,Product 26,3,Sport,0.74
1,2,1079,2024-01-03 17:45:43,16,9172.599609,1,paid,Product 16,3,Sport,1.29
2,3,1041,2024-01-06 22:46:38,25,9035.259766,2,paid,Product 25,3,Sport,0.20


### Задание 3. Посчитать выручку по комбинации (user_id, category_id).

In [4]:
# Вычисляем выручку с каждой строки (цена * количество)
if 'price' in df_merged.columns and 'quantity' in df_merged.columns:
    df_merged['revenue'] = df_merged['price'] * df_merged['quantity']
else:
    df_merged['revenue'] = 0

# Группируем по пользователю и категории, суммируем выручку
revenue_by_user_cat = df_merged.groupby(['user_id', 'category_id'], as_index=False)['revenue'].sum()

print("Выручка по комбинациям (user_id, category_id):")
display(revenue_by_user_cat.head())

Выручка по комбинациям (user_id, category_id):


,user_id,category_id,revenue
0,1001,1,1.079367e+06
1,1001,2,5.109530e+05
2,1001,3,1.212973e+06
3,1001,4,3.941912e+05
4,1001,5,1.238283e+06


### Задание 4. Найти топ-5 категорий для каждого пользователя по выручке.

In [5]:
# Сортируем данные: сначала по user_id (по возрастанию), затем по выручке (по убыванию)
revenue_by_user_cat = revenue_by_user_cat.sort_values(by=['user_id', 'revenue'], ascending=[True, False])

# Для каждого пользователя оставляем только 5 первых (самых прибыльных) категорий
top5_categories_per_user = revenue_by_user_cat.groupby('user_id').head(5).reset_index(drop=True)

print("Топ-5 категорий по выручке для каждого пользователя:")
display(top5_categories_per_user.head(10))

Топ-5 категорий по выручке для каждого пользователя:


,user_id,category_id,revenue
0,1001,5,1.238283e+06
1,1001,3,1.212973e+06
2,1001,1,1.079367e+06
3,1001,2,5.109530e+05
4,1001,4,3.941912e+05
5,1002,5,9.024817e+05
6,1002,3,7.036556e+05
7,1002,1,4.306263e+05
8,1002,2,4.098305e+05
9,1002,4,3.834103e+05


### Задание 5. Сохранить результат в формате parquet.

In [6]:
out_path = f"{data_dir}/top5_user_categories.parquet"

try:
    # Метод to_parquet требует установленной библиотеки pyarrow или fastparquet
    top5_categories_per_user.to_parquet(out_path, index=False)
    print(f"✅ Результат успешно сохранен в файл: {out_path}")
except ImportError:
    print("❌ Ошибка: для сохранения в Parquet установите библиотеку pyarrow (pip install pyarrow)")
except Exception as e:
    print(f"❌ Произошла ошибка при сохранении: {e}")

✅ Результат успешно сохранен в файл: data/top5_user_categories.parquet
